# 01 · Veriyi inceleme ve eğitim setlerini hazırlama

Gösterge: aylık ortalama TL vadesiz, TL vadeli ve kredi bakiyesi. KMH krediye dahildir;
kart borcu bu kredi tanımına eklenmez. Müşteri devirleri ve giriş/çıkışlar hazırlanmış şube
paneline yansımıştır. İlk çalıştırma tabloları inceler; veri üretimi ve hazırlık ayrı ayarlardır.


In [ ]:
from pathlib import Path
import sys
_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "python_kodlari/ayarlar").is_dir() and (p / "notebooks/jupyter_araclari.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Projenin tamamını indirin; notebook’u projenin notebooks klasöründen açın.")
for _folder in [ROOT / "python_kodlari", ROOT / "notebooks"]:
    if str(_folder) not in sys.path:
        sys.path.insert(0, str(_folder))
from IPython.display import display, Markdown
from jupyter_araclari import (read_json, read_table, latest_run, fresh_run, record_run,
    result_path, require_files, prepare_configs, source_input, run_python, extract_bundle,
    snapshot, current_decision, qualified_selection)
print("Proje:", ROOT)
print("Kernel Python:", sys.executable)


In [ ]:
import pandas as pd
VERI_KLASORU = ROOT / "outputs/banka_test_verisi_v4"
VERI_V3_KLASORU = ROOT / "outputs/banka_test_verisi_v3"
MUSTERI_VERISINI_YENIDEN_URET = False
EGITIM_SETINI_HAZIRLA = False


## İsteğe bağlı: müşteri hareketlerini yeniden üret

Bu hücre yalnızca tam V3 müşteri kaynakları varsa çalışır. Bu büyük müşteri dosyaları Git’te
yoktur; şube panelini kullanmak için bu adımı açmak gerekmez. Üretimden sonra bağımsız günlük
ortalama / devir / stok mutabakatı kontrolü yapılır.


In [ ]:
if MUSTERI_VERISINI_YENIDEN_URET:
    require_files(*(VERI_V3_KLASORU / n for n in ["musteriler.csv", "musteri_aylik.csv", "subeler.csv", "bolgeler_sentetik.csv", "veri_sozlugu.csv"]))
    generated = fresh_run("banka_test_verisi_v4")
    run_python("test_verisi_v4_uret.py", "--source", VERI_V3_KLASORU, "--output", generated)
    run_python("test_verisi_v4_dogrula.py", "--source", VERI_V3_KLASORU, "--output", generated)
    record_run("banka_test_verisi_v4", generated)
    VERI_KLASORU = generated
else:
    print("Hazır sentetik şube paneli kullanılacak.")


## Şube-ay verisi ve kalite kontrolü

Kendi verin için dosya yolunu değiştir. 03’teki sözlükte de gerçek anahtarları ve alan rollerini
eşleştir. Aşağıdaki örnek kontrol mevcut panelin şube / dönem sütunlarını kullanır.


In [ ]:
panel = read_table(VERI_KLASORU / "sube_model_girdileri_aylik.csv", dtype={"sube_id": str, "donem": str})
display(panel.head())
display(pd.DataFrame({"satır": [len(panel)], "şube": [panel.sube_id.nunique()], "ay": [panel.donem.nunique()], "ilk_ay": [panel.donem.min()], "son_ay": [panel.donem.max()]}))
targets = ["tl_vadesiz_ort_bakiye_tl", "tl_vadeli_ort_bakiye_tl", "kredi_ort_bakiye_tl"]
assert not panel[["sube_id", "donem"]].duplicated().any(), "Aynı şube-ay birden fazla kez var."
assert panel[targets].notna().all().all(), "Gösterge eksiği var."
assert panel[targets].ge(0).all().all(), "Negatif bakiye var."
display(panel.groupby(["donem", "buyuk_sehir_grubu"]).agg(sube=("sube_id", "nunique")).reset_index().tail(8))


In [ ]:
monthly = panel.groupby("donem")[targets].sum()
display(monthly.tail(6))
ax = monthly.plot(title="Banka geneli aylık ortalama bakiyeler · sentetik", figsize=(10, 4))
ax.set_ylabel("TL")


## Zaman ayrımı

Özellik ayı `t`, hedef ayı `t+1`’dir. TRAIN / VALIDATION / TEST tarih sırasıyla ayrılır;
rastgele satır bölünmez. Mevcut tarihsel V4 deneyinde 3.400 / 600 / 600 etiketli satır vardır.
Son ayın 200 şubesi Ocak 2027 tahmin girdisidir. Bu hazırlanmış tablolar eğitim için yeniden
üretilmeden de incelenebilir.


In [ ]:
split_rows = []
for split in ["train", "validation", "test"]:
    frame = read_table(ROOT / "model_dashboard/dist/downloads" / f"v4_{split}.csv")
    split_rows.append({"split": split.upper(), "satır": len(frame), "ilk_özellik_ayı": frame.ozellik_donemi.min(), "son_özellik_ayı": frame.ozellik_donemi.max(), "ilk_hedef_ayı": frame.hedef_donemi.min(), "son_hedef_ayı": frame.hedef_donemi.max()})
display(pd.DataFrame(split_rows))
assert split_rows[0]["son_hedef_ayı"] < split_rows[1]["ilk_hedef_ayı"]
assert split_rows[1]["son_hedef_ayı"] < split_rows[2]["ilk_hedef_ayı"]


In [ ]:
if EGITIM_SETINI_HAZIRLA:
    require_files(*(VERI_KLASORU / n for n in ["manifest.json", "sube_model_girdileri_aylik.csv", "sube_stok_mutabakat.csv", "sube_aylik.csv"]))
    from v4_egitim_seti_hazirla import run as prepare_training
    prepared = fresh_run("model_karsilastirma_v4_veri")
    prepare_training(VERI_KLASORU, prepared)
    record_run("model_karsilastirma_v4_veri", prepared)
    display(read_table(prepared / "veri/zaman_ayrimi.csv"))
    print("02 notebook’unun eğitim girdisi:", prepared)
else:
    print("Eğitim seti yazılmadı. Üretmek için EGITIM_SETINI_HAZIRLA = True yapın.")


Sonraki adım: tahmin deneyleri için **02**, güncel emsal kümeleme akışı için **03**.
Bu V4 hazırlayıcı mevcut sentetik 26 ay protokolüne özgüdür; gerçek veri için 03’teki dinamik
sözlük akışını kullanın.
